# 01. 기초 — 인과적 fast-weight 상태를 직접 만들기

이 노트북은 논문 **Fast Weight Attention for Continual Learning**의 출발점인
선형 어텐션의 recurrent state와 `read-after-write` 인덱싱을 작은 NumPy 예제로 확인한다.

- 원문: <https://arxiv.org/abs/2608.27763>
- 목표: 명시적 prefix 합과 recurrent update의 동치, shifted pair의 인과성, 정규화 read의 차이를 확인한다.
- 범위: 논문 규모 언어 모델을 재현하는 코드가 아니라 수식과 텐서 shape를 검증하는 toy reproduction이다.

## 1. 핵심 표기

시점 $t$의 fast-weight matrix를 $S_t\in\mathbb{R}^{d\times d_v}$라 하자.
논문의 next-latent 정렬에서는 다음과 같이 한 칸 이동한 causal write pair를 사용한다.

$$x_t=\phi(k_{t-1}),\qquad y_t=v_t,\qquad x_1=0.$$

가장 단순한 additive write는 $S_t=S_{t-1}+x_ty_t^\top$이고,
read-after-write 출력은 $o_t=S_t^\top\phi(q_t)$이다. 과거와 현재의 write만
사용하므로 미래 값을 바꿔도 prefix 출력은 달라지지 않아야 한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(260827763)

def elu_plus_one(x):
    # 양수 feature map: ELU(x) + 1.
    return np.where(x >= 0.0, x + 1.0, np.exp(x))

T, d, d_v = 18, 4, 3
keys = rng.normal(size=(T, d))
queries = keys + 0.05 * rng.normal(size=(T, d))
values = rng.normal(size=(T, d_v))

key_features = elu_plus_one(keys)
query_features = elu_plus_one(queries)
x_shifted = np.zeros_like(key_features)
x_shifted[1:] = key_features[:-1]

print("x:", x_shifted.shape, "values:", values.shape)
print("state shape:", (d, d_v), "output shape:", (T, d_v))

## 2. 명시적 prefix 합과 recurrence 비교

아래 두 구현은 계산 순서만 다르다. `explicit_prefix`는 매 시점 처음부터 합하고,
`recurrent_prefix`는 이전 상태를 한 번만 갱신한다. 실제 장기 시퀀스에서 중요한 것은
후자의 고정 크기 상태다.

In [ ]:
def explicit_prefix(x, y, q):
    outputs = []
    states = []
    for t in range(len(x)):
        state_t = sum(
            (np.outer(x[j], y[j]) for j in range(t + 1)),
            start=np.zeros((x.shape[1], y.shape[1])),
        )
        states.append(state_t)
        outputs.append(state_t.T @ q[t])
    return np.stack(outputs), np.stack(states)

def recurrent_prefix(x, y, q):
    state = np.zeros((x.shape[1], y.shape[1]))
    outputs, states = [], []
    for t in range(len(x)):
        state = state + np.outer(x[t], y[t])
        states.append(state.copy())
        outputs.append(state.T @ q[t])
    return np.stack(outputs), np.stack(states)

out_explicit, states_explicit = explicit_prefix(x_shifted, values, query_features)
out_recurrent, states_recurrent = recurrent_prefix(x_shifted, values, query_features)

max_output_error = np.max(np.abs(out_explicit - out_recurrent))
max_state_error = np.max(np.abs(states_explicit - states_recurrent))
print(f"maximum output error: {max_output_error:.3e}")
print(f"maximum state error : {max_state_error:.3e}")

np.testing.assert_allclose(out_explicit, out_recurrent, atol=1e-12)
np.testing.assert_allclose(states_explicit, states_recurrent, atol=1e-12)

## 3. 미래 교란으로 인과성 검사

시점 `cutoff` 뒤의 value를 크게 바꾼다. 올바른 causal recurrence라면 `cutoff`까지의
출력은 동일하고, 그 뒤만 달라진다. 이 검사는 구현에서 실수로 미래 tensor를 참조하는
오류를 빠르게 찾는 작은 단위 테스트다.

In [ ]:
cutoff = 9
future_changed = values.copy()
future_changed[cutoff + 1:] += 100.0
out_changed, _ = recurrent_prefix(x_shifted, future_changed, query_features)

prefix_delta = np.max(np.abs(out_changed[: cutoff + 1] - out_recurrent[: cutoff + 1]))
suffix_delta = np.max(np.abs(out_changed[cutoff + 1:] - out_recurrent[cutoff + 1:]))
print(f"prefix delta: {prefix_delta:.3e}")
print(f"suffix delta: {suffix_delta:.3e}")

assert prefix_delta == 0.0
assert suffix_delta > 1.0

## 4. denominator-free read와 정규화 read

양수 feature map에서는 누적 feature $z_t=\sum_{j\le t}x_j$를 두고
$S_t^\top\phi(q_t)/(z_t^\top\phi(q_t)+\epsilon)$처럼 정규화할 수 있다.
두 출력은 scale이 다르며 서로 같은 연산이 아니다. 특히 signed feature에서는 분모가
0이 되거나 부호가 바뀔 수 있어, 논문은 명시적인 안전 분모·clamp가 없다면
denominator-free 해석을 기본으로 둔다.

In [ ]:
def normalized_recurrent(x, y, q, eps=1e-8):
    state = np.zeros((x.shape[1], y.shape[1]))
    normalizer = np.zeros(x.shape[1])
    outputs, denominators = [], []
    for t in range(len(x)):
        state += np.outer(x[t], y[t])
        normalizer += x[t]
        denominator = float(normalizer @ q[t] + eps)
        outputs.append((state.T @ q[t]) / denominator)
        denominators.append(denominator)
    return np.stack(outputs), np.asarray(denominators)

out_normalized, denominators = normalized_recurrent(
    x_shifted, values, query_features
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(np.linalg.norm(out_recurrent, axis=1), label="denominator-free")
axes[0].plot(np.linalg.norm(out_normalized, axis=1), label="normalized")
axes[0].set(title="output norm", xlabel="time", ylabel="L2 norm")
axes[0].legend()
axes[1].plot(denominators, color="tab:green")
axes[1].set(title="positive-feature denominator", xlabel="time", ylabel="value")
fig.tight_layout()
plt.show()

assert np.all(np.isfinite(out_normalized))
assert np.all(denominators > 0.0)

## 정리와 다음 단계

1. prefix outer-product 합은 고정 크기 recurrent state로 정확히 계산할 수 있다.
2. shifted pair는 `k[t-1]`과 `v[t]`를 묶고 첫 write feature는 0으로 둔다.
3. 미래 교란 검사는 causal mask·인덱스 오류를 잡는 최소 검증이다.
4. 다음 노트북에서는 additive write 대신 residual regression update를 사용해
   Falcon-1과 Falcon-2를 비교한다.